In [1]:
import os

In [2]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataProcessingConfig:
    root_dir: Path
    movies: Path
    ratings: Path
    tags: Path

In [6]:
from mlProject.utils.common import read_yaml, create_directories
from mlProject.constants import *

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH
            ) -> None:
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_processing_config(self) -> DataProcessingConfig:

        config  = self.config.data_processing

        create_directories([config.root_dir])

        data_processing_config = DataProcessingConfig(
            movies= config.movies,
            tags= config.tags,
            ratings= config.ratings,
            root_dir= config.root_dir
        )

        return data_processing_config

In [107]:
import pandas as pd

config = ConfigurationManager()
data_processing_config = config.get_data_processing_config()

movies_df = pd.read_csv(data_processing_config.movies)
ratings_df = pd.read_csv(data_processing_config.ratings)
tags_df = pd.read_csv(data_processing_config.tags)
print(data_processing_config)

[2025-01-18 09:29:14,001: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2025-01-18 09:29:14,154: INFO: common: Yaml file : params.yaml loaded successfully]
[2025-01-18 09:29:14,210: INFO: common: Yaml file : schema.yaml loaded successfully]
[2025-01-18 09:29:14,263: INFO: common: Created directory at: artifacts]
[2025-01-18 09:29:14,327: INFO: common: Created directory at: artifacts/data_transformation]


DataProcessingConfig(root_dir='artifacts/data_transformation', movies='artifacts/data_ingestion/ml-latest-small/movies.csv', ratings='artifacts/data_ingestion/ml-latest-small/ratings.csv', tags='artifacts/data_ingestion/ml-latest-small/tags.csv')


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Load datasets
movies_df = pd.read_csv(data_processing_config.movies)
ratings_df = pd.read_csv(data_processing_config.ratings)
tags_df = pd.read_csv(data_processing_config.tags)

# Select 5 users who have rated the most movies
top_5_users = ratings_df['userId'].value_counts().head(50).index
ratings_df = ratings_df[ratings_df['userId'].isin(top_5_users)]

# Filter movies and tags to only include those related to these users
relevant_movies = ratings_df['movieId'].unique()
movies_df = movies_df[movies_df['movieId'].isin(relevant_movies)]
tags_df = tags_df[tags_df['userId'].isin(top_5_users)]

def calculate_user_genre_ratings(ratings_df, movies_df):
    # Create genre columns
    genres = movies_df['genres'].str.get_dummies('|')
    movies_with_genres = pd.concat([movies_df[['movieId']], genres], axis=1)
    
    # Merge ratings with movies and genres
    ratings_with_genres = ratings_df.merge(movies_with_genres, on='movieId')
    
    # Calculate average rating per genre per user
    genre_columns = genres.columns
    user_genre_ratings = []
    
    for user_id in ratings_with_genres['userId'].unique():
        user_ratings = ratings_with_genres[ratings_with_genres['userId'] == user_id]
        genre_avgs = {}
        genre_avgs['userId'] = user_id
        
        for genre in genre_columns:
            genre_movies = user_ratings[user_ratings[genre] == 1]
            genre_avgs[f'{genre}_avg_rating'] = genre_movies['rating'].mean() if len(genre_movies) > 0 else 0
            
        user_genre_ratings.append(genre_avgs)
    
    return pd.DataFrame(user_genre_ratings)

def prepare_user_features(ratings_df, movies_df, tags_df):
    # Basic user statistics
    rating_stats = ratings_df.groupby('userId').agg({
        'rating': ['mean', 'std', 'count']
    }).fillna(0)
    rating_stats.columns = ['avg_rating', 'std_rating', 'rating_count']
    rating_stats = rating_stats.reset_index()
    
    # Tag statistics
    tag_stats = tags_df.groupby('userId').agg({
        'tag': 'count'
    }).reset_index()
    tag_stats.columns = ['userId', 'tag_count']
    
    # Genre rating averages
    genre_ratings = calculate_user_genre_ratings(ratings_df, movies_df)
    
    # Combine all user features
    user_features = rating_stats.merge(tag_stats, on='userId', how='left')
    user_features = user_features.merge(genre_ratings, on='userId', how='left')
    user_features = user_features.fillna(0)
    
    # Save user IDs before normalization
    user_ids = user_features['userId']
    
    # Standardize all features
    feature_columns = [col for col in user_features.columns if col != 'userId']
    user_features_normalized = StandardScaler().fit_transform(user_features[feature_columns])
    
    return user_features_normalized, user_features

def prepare_movie_features(movies_df, tags_df):
    # Genre features
    genres = movies_df['genres'].str.get_dummies('|')
    
    # Extract year
    movies_df['year'] = movies_df['title'].str.extract('(\d{4})', expand=False)
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
    year_normalized = StandardScaler().fit_transform(movies_df[['year']].fillna(movies_df['year'].mean()))
    
    # Tag features
    movie_tags = tags_df.groupby('movieId')['tag'].agg(lambda x: ' '.join(x)).reset_index()
    movie_tags = movies_df[['movieId']].merge(movie_tags, on='movieId', how='left')
    movie_tags['tag'] = movie_tags['tag'].fillna('')
    
    # Reduce TF-IDF features for the sample
    tfidf = TfidfVectorizer(max_features=50, stop_words='english')
    tag_features = tfidf.fit_transform(movie_tags['tag']).toarray()
    
    # Combine all features
    movie_features = np.hstack([
        genres.values,
        year_normalized,
        tag_features
    ])
    
    return movie_features, tfidf

def prepare_training_data(ratings_df, movie_features, user_features):
    user_encoder = LabelEncoder()
    movie_encoder = LabelEncoder()
    
    ratings_df['user_encoded'] = user_encoder.fit_transform(ratings_df['userId'])
    ratings_df['movie_encoded'] = movie_encoder.fit_transform(ratings_df['movieId'])
    
    X_user = user_features[ratings_df['user_encoded']]
    X_movie = movie_features[ratings_df['movie_encoded']]
    y = ratings_df['rating'].values
    
    return X_user, X_movie, y, user_encoder, movie_encoder

# Execute the pipeline
movie_features, tfidf = prepare_movie_features(movies_df, tags_df)
user_features, user_features_df = prepare_user_features(ratings_df, movies_df, tags_df)
X_user, X_movie, y, user_encoder, movie_encoder = prepare_training_data(ratings_df, movie_features, user_features)


# Display information about the processed data
print("Dataset Overview:")
print(f"Number of users: {len(top_5_users)}")
print(f"Number of movies: {len(movies_df)}")
print(f"Number of ratings: {len(ratings_df)}")

print("\nSample of User Features:")
print("\nUser Statistics:")
print(user_features_df[['userId', 'avg_rating', 'std_rating', 'rating_count', 'tag_count']].to_string())

print("\nSample of Genre Preferences for first user:")
genre_columns = [col for col in user_features_df.columns if '_avg_rating' in col]
print(user_features_df[['userId'] + genre_columns].iloc[20].to_string())

print("\nFeature Dimensions:")
print(f"User features shape: {user_features.shape}")
print(f"Movie features shape: {movie_features.shape}")

# Split data
train_size = 0.8
indices = np.arange(len(y))
train_indices, val_indices = train_test_split(indices, train_size=train_size, random_state=42)

# Create final sets
train_user_features = X_user[train_indices]
train_movie_features = X_movie[train_indices]
train_ratings = y[train_indices]

val_user_features = X_user[val_indices]
val_movie_features = X_movie[val_indices]
val_ratings = y[val_indices]

print("\nTraining/Validation Split:")
print(f"Training samples: {len(train_indices)}")
print(f"Validation samples: {len(val_indices)}")

## OOP


In [8]:
# base feature transformer
from abc import ABC, abstractmethod
import pandas as pd
from typing import Optional

class FeatureExtractor(ABC):

    @abstractmethod
    def generate_features(self, ratings_df: Optional[pd.DataFrame], movies_df: Optional[pd.DataFrame], tags_df: Optional[pd.DataFrame] ) -> pd.DataFrame:
        pass

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import numpy as np

class MovieFeatureExtractor(FeatureExtractor):   

    def generate_features(self, movies_df : pd.DataFrame, tags_df: pd.DataFrame) -> np.ndarray:
        genres = movies_df['genres'].str.get_dummies('|')
    
        # Extract year
        movies_df['year'] = movies_df['title'].str.extract('(\d{4})', expand=False)
        movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
        year_normalized = StandardScaler().fit_transform(movies_df[['year']].fillna(movies_df['year'].mean()))
        
        # Tag features
        movie_tags = tags_df.groupby('movieId')['tag'].agg(lambda x: ' '.join(x)).reset_index()
        movie_tags = movies_df[['movieId']].merge(movie_tags, on='movieId', how='left')
        movie_tags['tag'] = movie_tags['tag'].fillna('')
        
        # Reduce TF-IDF features for the sample
        tfidf = TfidfVectorizer(max_features=50, stop_words='english')
        tag_features = tfidf.fit_transform(movie_tags['tag']).toarray()
        
        # Combine all features
        movie_features = np.hstack([
            genres.values,
            year_normalized,
            tag_features
        ])
        
        return movie_features

In [25]:
class UserFeatureExtractor(FeatureExtractor):
    """
    Extracts user features from ratings and movies data.

    Attributes:
        None

    Methods:
        generate_features: Generates user features based on ratings and movies data.

    """

    def calculate_user_genre_ratings(self, ratings_df, movies_df):
        # Create genre columns
        genres = movies_df['genres'].str.get_dummies('|')
        movies_with_genres = pd.concat([movies_df[['movieId']], genres], axis=1)
        
        # Merge ratings with movies and genres
        ratings_with_genres = ratings_df.merge(movies_with_genres, on='movieId')
        
        # Calculate average rating per genre per user
        genre_columns = genres.columns
        user_genre_ratings = []
        
        for user_id in ratings_with_genres['userId'].unique():
            user_ratings = ratings_with_genres[ratings_with_genres['userId'] == user_id]
            genre_avgs = {}
            genre_avgs['userId'] = user_id
            
            for genre in genre_columns:
                genre_movies = user_ratings[user_ratings[genre] == 1]
                genre_avgs[f'{genre}_avg_rating'] = genre_movies['rating'].mean() if len(genre_movies) > 0 else 0
                
            user_genre_ratings.append(genre_avgs)
        
        return pd.DataFrame(user_genre_ratings)

    def generate_features(self, ratings_df: pd.DataFrame, movies_df: pd.DataFrame, tags_df: pd.DataFrame) -> pd.DataFrame:
        # Basic user statistics
        rating_stats = ratings_df.groupby('userId').agg({
            'rating': ['mean', 'std', 'count']
        }).fillna(0)
        rating_stats.columns = ['avg_rating', 'std_rating', 'rating_count']
        rating_stats = rating_stats.reset_index()
        
        # Tag statistics
        tag_stats = tags_df.groupby('userId').agg({
            'tag': 'count'
        }).reset_index()
        tag_stats.columns = ['userId', 'tag_count']
        
        # Genre rating averages
        genre_ratings = self.calculate_user_genre_ratings(ratings_df, movies_df)
        
        # Combine all user features
        user_features = rating_stats.merge(tag_stats, on='userId', how='left')
        user_features = user_features.merge(genre_ratings, on='userId', how='left')
        user_features = user_features.fillna(0)
        
        # Save user IDs before normalization
        user_ids = user_features['userId']
        
        # Standardize all features
        feature_columns = [col for col in user_features.columns if col != 'userId']
        user_features_normalized = StandardScaler().fit_transform(user_features[feature_columns])
        
        return user_features_normalized, user_features

In [36]:
from typing import Tuple
from mlProject import logger
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

class CSVDataPreprocessor:
    def __init__(self, config: DataProcessingConfig) -> None:
        self.movie_feature_extractor = MovieFeatureExtractor()
        self.user_feature_extractor = UserFeatureExtractor()
        self.config = config
        self.movies_df = pd.read_csv(self.config.movies)
        self.ratings_df = pd.read_csv(self.config.ratings)
        self.tags_df = pd.read_csv(self.config.tags)

        # Select 5 users who have rated the most movies
        self.top_5_users = self.ratings_df['userId'].value_counts().head(50).index
        self.ratings_df = self.ratings_df[self.ratings_df['userId'].isin(self.top_5_users)]

        # Filter movies and tags to only include those related to these users
        self.relevant_movies = self.ratings_df['movieId'].unique()
        self.movies_df = self.movies_df[self.movies_df['movieId'].isin(self.relevant_movies)]
        self.tags_df = self.tags_df[self.tags_df['userId'].isin(self.top_5_users)]
    
    def prepare_training_data(self, ratings_df, movie_features, user_features):
        user_encoder = LabelEncoder()
        movie_encoder = LabelEncoder()
        
        ratings_df['user_encoded'] = user_encoder.fit_transform(self.ratings_df['userId'])
        ratings_df['movie_encoded'] = movie_encoder.fit_transform(self.ratings_df['movieId'])
        
        X_user = user_features[ratings_df['user_encoded']]
        X_movie = movie_features[ratings_df['movie_encoded']]
        y = ratings_df['rating'].values
        
        return X_user, X_movie, y, user_encoder, movie_encoder
    
    def process_data(self) -> pd.DataFrame:
        
        logger.info(f"Extracting Movies features...⏳")
        movie_features = self.movie_feature_extractor.generate_features(self.movies_df, self.tags_df)
        logger.info(f"Extracting Movies features completed ✅ ")
        logger.info(f"Extracting User features...⏳")
        user_features, user_features_df = self.user_feature_extractor.generate_features(self.ratings_df, self.movies_df, self.tags_df)
        logger.info(f"Extracting User features completed ✅")
        logger.info(f"Merging features with ratings...⏳")
        X_user, X_movie, y, _, _ = self.prepare_training_data(self.ratings_df, movie_features, user_features)
        logger.info(f"Merging features with ratings completed ✅")

        return X_user, X_movie, y

    def train_validation_split(self, X_user, X_movie, y) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        train_size = 0.8
        indices = np.arange(len(y))
        train_indices, val_indices = train_test_split(indices, train_size=train_size, random_state=42)

        # Create final sets
        train_user_features = X_user[train_indices]
        train_movie_features = X_movie[train_indices]
        train_ratings = y[train_indices]

        val_user_features = X_user[val_indices]
        val_movie_features = X_movie[val_indices]
        val_ratings = y[val_indices]

        return train_user_features, train_movie_features, train_ratings, val_user_features, val_movie_features, val_ratings
    

In [37]:
try:
    config = ConfigurationManager()
    get_data_processing_config = config.get_data_processing_config()
    data_preprocessor = CSVDataPreprocessor(config=get_data_processing_config)
    x_user, x_movie, y = data_preprocessor.process_data()
    X_train_user, X_train_movie, y_train_rating, X_val_user, X_val_movie, y_val_rating = data_preprocessor.train_validation_split(x_user, x_movie, y)
except Exception as e:
    logger.exception(f"Oops😟! An error occured: {e} ")


[2021-01-01 03:34:31,470: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2021-01-01 03:34:31,510: INFO: common: Yaml file : params.yaml loaded successfully]
[2021-01-01 03:34:31,560: INFO: common: Yaml file : schema.yaml loaded successfully]
[2021-01-01 03:34:31,564: INFO: common: Created directory at: artifacts]
[2021-01-01 03:34:31,573: INFO: common: Created directory at: artifacts/data_transformation]


[2021-01-01 03:34:32,124: INFO: 3558107034: Extracting Movies features...⏳]
[2021-01-01 03:34:34,045: INFO: 3558107034: Extracting Movies features completed ✅ ]
[2021-01-01 03:34:34,049: INFO: 3558107034: Extracting User features...⏳]
[2021-01-01 03:34:38,805: INFO: 3558107034: Extracting User features completed ✅]
[2021-01-01 03:34:38,807: INFO: 3558107034: Merging features with ratings...⏳]
[2021-01-01 03:34:38,957: INFO: 3558107034: Merging features with ratings completed ✅]


# User Features


## Step 1: Aggregate Ratings


In [ ]:
# scale training data
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled    = y_train

scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

scalerTarget = MinMaxScaler((-1, 1))
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))
#ynorm_test = scalerTarget.transform(y_test.reshape(-1, 1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))

In [ ]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

model.summary()

In [ ]:
# model

from tensorflow.keras import layers, Model
import tensorflow as tf
import numpy as np
from typing import Tuple, List

class RecommenderNet(Model):
    def __init__(self, user_shape: int, movie_shape: int):
        super(RecommenderNet, self).__init__()
        
        # User tower layers
        self.user_input = layers.Input(shape=(user_shape,), name="user_input")
        self.user_dense1 = layers.Dense(64, activation="relu")
        self.user_dense2 = layers.Dense(32, activation="relu")
        
        # Movie tower layers
        self.movie_input = layers.Input(shape=(movie_shape,), name="movie_input")
        self.movie_dense1 = layers.Dense(64, activation="relu")
        self.movie_dense2 = layers.Dense(32, activation="relu")
        
        # Combined layers
        self.combined_dense = layers.Dense(64, activation="relu")
        self.output_layer = layers.Dense(1, activation="linear", name="output")
        
    def call(self, inputs):
        user_input, movie_input = inputs
        
        # User tower
        x1 = self.user_dense1(user_input)
        x1 = self.user_dense2(x1)
        
        # Movie tower
        x2 = self.movie_dense1(movie_input)
        x2 = self.movie_dense2(x2)
        
        # Combine towers
        combined = layers.concatenate([x1, x2])
        combined = self.combined_dense(combined)
        
        return self.output_layer(combined)
    
    def build_graph(self):
        """Create model graph for visualization"""
        model = Model(
            inputs=[self.user_input, self.movie_input],
            outputs=self.call([self.user_input, self.movie_input])
        )
        return model

class RecommenderTrainer:
    def __init__(
        self,
        user_shape: int,
        movie_shape: int,
        learning_rate: float = 0.001,
        batch_size: int = 32,
        epochs: int = 10
    ):
        self.model = RecommenderNet(user_shape, movie_shape)
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.history = None
        
    def compile_model(self):
        """Compile the model with specified parameters"""
        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        self.model.compile(
            optimizer=optimizer,
            loss='mse',
            metrics=['mae']
        )
        
    def train(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray,
        validation_split: float = 0.2,
        callbacks: List = None
    ):
        """Train the model"""
        self.compile_model()
        self.history = self.model.fit(
            [X_user, X_movie],
            y,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_split=validation_split,
            callbacks=callbacks
        )
        return self.history
    
    def predict(self, X_user: np.ndarray, X_movie: np.ndarray) -> np.ndarray:
        """Make predictions"""
        return self.model.predict([X_user, X_movie])
    
    def evaluate(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray
    ) -> Tuple[float, float]:
        """Evaluate the model"""
        return self.model.evaluate([X_user, X_movie], y)
    
    def save_model(self, path: str):
        """Save the model"""
        self.model.save(path)
    
    @staticmethod
    def load_model(path: str):
        """Load a saved model"""
        return tf.keras.models.load_model(path)

# Usage example:
"""
# Initialize trainer
trainer = RecommenderTrainer(
    user_shape=X_user_np.shape[1],
    movie_shape=X_movie_np.shape[1],
    learning_rate=0.001,
    batch_size=32,
    epochs=10
)

# Define callbacks if needed
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
]

# Train the model
history = trainer.train(
    X_user_np,
    X_movie_np,
    y,
    validation_split=0.2,
    callbacks=callbacks
)

# Make predictions
predictions = trainer.predict(X_user_test, X_movie_test)

# Evaluate model
loss, mae = trainer.evaluate(X_user_test, X_movie_test, y_test)

# Save model
trainer.save_model('recommender_model.h5')

# Load model later
loaded_model = RecommenderTrainer.load_model('recommender_model.h5')
"""